In [0]:
# Group tasks by batch_id
batches = {}
for task in tasks:
    batches.setdefault(task.batch_id, []).append(task)

max_batch_workers = len(batches) # e.g., 5 parallel batches

def run_batch(batch_id, batch_tasks: list) -> list:
    """Run every table in the batch in PARALLEL using an inner ThreadPool."""
    batch_results = []
    print(f"[Batch {batch_id}] Starting {len(batch_tasks)} table(s) in PARALLEL...")
    
    # Inner ThreadPool: Spawns a worker for every table in this specific batch
    with ThreadPoolExecutor(max_workers=len(batch_tasks)) as inner_executor:
        future_to_task = {
            inner_executor.submit(run_one, task): task 
            for task in batch_tasks
        }
        
        for future in as_completed(future_to_task):
            task = future_to_task[future]
            try:
                batch_results.append(future.result())
            except Exception as exc:
                print(f"Task {task.source_object_name} (Config ID: {task.config_id}) failed: {exc}")
                batch_results.append({
                    "config_id": task.config_id,
                    "run_id":    None,
                    "status":    AUDIT_STATUS_FAILED,
                    "rows_read": 0,
                    "error":     str(exc),
                })
                
    return batch_results

print(f"\nStarting {len(tasks)} tasks across {len(batches)} parallel batch(es)...")

# Outer ThreadPool: Spawns a worker for each batch
with ThreadPoolExecutor(max_workers=max_batch_workers) as executor:
    future_to_batch = {
        executor.submit(run_batch, batch_id, batch_tasks): batch_id
        for batch_id, batch_tasks in batches.items()
    }

    for future in as_completed(future_to_batch):
        batch_id = future_to_batch[future]
        try:
            results.extend(future.result())
        except Exception as exc:
            print(f"Batch {batch_id} failed with exception: {exc}")
            for task in batches[batch_id]:
                results.append({
                    "config_id": task.config_id,
                    "run_id":    None,
                    "status":    AUDIT_STATUS_FAILED,
                    "rows_read": 0,
                    "error":     str(exc),
                })